## Haruhi Cleaning Script

*Source: <https://huggingface.co/datasets/silk-road/ChatHaruhi-Expand-118K>

### Load dataset from file

In [ ]:
# Load and Clean the Dataset
# {"user_role": "旅行者", "user_question": "「我特意准备的点心,尝一尝吧。」", "agent_role": "雷电将军", "agent_response": "「哦？你特意为我准备的点心吗？看来你对我很了解啊。好吧，我就尝一尝。」", "question_source": "story", "more_dialogues": [], "agent_role_name_en": "raidenShogun"}

from pathlib import Path
import json

MIN_RESPONSE_LEN = 10
TARGET_AGENT_ROLES = ["钟离", "旅行者", "神里绫华", "派蒙", "凉宫", "胡桃", "春日", "雷电将军", "李云龙", "Sheldon"]

ORIGIN_HARUHI_DATA_PATH = Path(r"..\Dataset\Haruhi\Haruhi_54K_v1.jsonl")
OUTPUT_DATASET_PATH = Path(r".\data\raw\Haruhi\Haruhi_clean.jsonl")

OUTPUT_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)

lines = ORIGIN_HARUHI_DATA_PATH.read_text(encoding="utf-8").split("\n")
datasets: list[dict[str,str]] = []
user_roles: list[str] = []
agent_roles: list[str] = []

for line in lines:
    if not line:
        continue        
    data = json.loads(line)
    datasets.append(data)
    user_roles.append(data["user_role"])
    agent_roles.append(data["agent_role"])

user_roles_set = set(user_roles)
agent_roles_set = set(agent_roles)

print("user_roles: ", len(user_roles_set))
print("agent_roles: ", len(agent_roles_set))

anime_datasets = []

for item in datasets:
    if item["agent_role"] in TARGET_AGENT_ROLES:
        item["user_question"] = item["user_question"].removeprefix("「").removesuffix("」").replace("<br>", "")
        item["agent_response"] = item["agent_response"].removeprefix("「").removesuffix("」").replace("<br>", "")
        if len(item["agent_response"]) < MIN_RESPONSE_LEN:
            continue
            
        anime_datasets.append(item)

# output_dataset_path.write_text("\n".join([json.dumps(item, ensure_ascii=False) for item in anime_datasets]), encoding="utf-8")

user_roles:  1675
agent_roles:  299


### Detect English entries

In [4]:
import re

TARGET_FIELDS = ["agent_response"]


def is_english_sentence(text: str) -> bool:
    if not text or not text.strip():
        return False
    zh_chars = len(re.findall(r"[\u4e00-\u9fff]", text))
    en_chars = len(re.findall(r"[A-Za-z]", text))
    return en_chars >= 8 and en_chars > zh_chars * 2


english_candidates: list[dict[str, object]] = []

for idx, item in enumerate(anime_datasets):
    content = item.get("agent_response", None)
    if content and item.get("agent_role", None) == "Sheldon" and is_english_sentence(content):
        item["item_index"] = idx
        english_candidates.append(item)

print(f"Filtered dataset size: {len(anime_datasets)}")
print(f"English fields to translate: {len(english_candidates)}")
print("Preview:")
for i, row in enumerate(english_candidates[:5], start=1):
    print(
        f"{i}. idx={row['item_index']}, "
        f"agent_role={row['agent_role']}, text={row['agent_response']}"
    )

Filtered dataset size: 9273
English fields to translate: 1504
Preview:
1. idx=0, agent_role=Sheldon, text=Ah, Raj, your attempt at humor is as misguided as your assumption that bagels are exclusive to Mumbai. Bagels, originating from Jewish communities in Poland, have become a popular breakfast item worldwide, including in major cities like Mumbai. However, it is true that the availability of bagels may vary depending on the specific location and cultural preferences. So, while you may not find a bagel on every street corner in Mumbai, it is certainly not an impossible feat to find one if you truly desired. Now, if you'll excuse me, I need to recalibrate my sarcasm detector after that failed attempt at wit.
2. idx=1, agent_role=Sheldon, text=A more plausible explanation is that his work in robotics has made an amazing leap forward.
3. idx=2, agent_role=Sheldon, text=Inefficiency.
4. idx=3, agent_role=Sheldon, text=Again, time machine.
5. idx=4, agent_role=Sheldon, text=Well, Penny, you

### Translate English Item to Chinese

In [3]:
import os
from dataclasses import dataclass, field
from typing import Any, Dict, Optional, Sequence, Tuple

from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam as ChatMsgParam
from tqdm import tqdm

load_dotenv()

DEFAULT_API_KEY = os.getenv("OPENAI_API_KEY", "sk-PLACEHOLDER")
DEFAULT_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1")
DEFAULT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "qwen3-max-2025-09-23")

ROLE_BACKGROUND = ("语言高度理性化、学术化，惯用科学术语与严谨句式解构日常琐事，将简单对话升级为微型学术报告；他直言不讳、缺乏社交缓冲，常以\"事实陈述\"的姿态说出令人啼笑皆非的\"冒犯真理\""
"，却又在\"逗你玩呢\"的反转中流露笨拙的幽默；他执着于秩序与仪式，从三遍敲门到专属座位，言语间处处彰显对\"一致性\"的强迫式追求；然而在这层理性铠甲之下，偶尔吟唱的\"软软小猫\"或对友情的隐性依赖，"
"又让这份\"高智商低情商\"的表达多了几分反差萌与人性温度")
TRANSLATE_SYSTEM_PROMPT = f"你是动漫角色台词翻译专家。你将要翻译角色 Sheldon Cooper 的角色台词。将英文翻译成中文时，做到信、达、雅：忠实原意、表达自然、文风优雅，并保留角色口吻与人格特征（Sheldon 的说话风格: {ROLE_BACKGROUND}）。只输出译文本身。"

ConversationTurn = Tuple[str, Optional[str]]
HISTORY_SHOT = [("Bazinga!", "中计了吧！"), ("That's my spot. In an ever-changing world, it is a single point of consistency.", "那是我的专座。在这个变幻无常的世界里，那是我唯一的恒定之点。")]

def _build_messages(
    prompt: str,
    history: Optional[Sequence[ConversationTurn]] = None,
    system_prompt: Optional[str] = None,
) -> list[ChatMsgParam]:
    messages: list[ChatMsgParam] = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    if history:
        for user_msg, assistant_msg in history:
            messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})

    messages.append({"role": "user", "content": prompt})
    return messages


@dataclass(slots=True)
class SimpleLLMClient:
    model: str = DEFAULT_MODEL
    api_key: str = DEFAULT_API_KEY
    base_url: Optional[str] = DEFAULT_BASE_URL
    system_prompt: Optional[str] = TRANSLATE_SYSTEM_PROMPT
    extra_headers: Optional[Dict[str, str]] = None
    _client: OpenAI = field(init=False, repr=False)

    def __post_init__(self) -> None:
        self._client = OpenAI(
            api_key=self.api_key,
            base_url=self.base_url,
            default_headers=self.extra_headers if self.extra_headers else None,
        )

    def ask(
        self,
        prompt: str,
        history: Optional[Sequence[ConversationTurn]] = None,
        *,
        temperature: float = 0.4,
        top_p: float = 0.9,
        max_tokens: Optional[int] = 512,
        **kwargs: Any,
) -> str:
        messages = _build_messages(prompt=prompt, history=history, system_prompt=self.system_prompt)
        response = self._client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
            stream=False,
            extra_body={"enable_thinking": False},
            **kwargs,
        )
        choice = response.choices[0]
        if hasattr(choice, "message") and getattr(choice.message, "content", None):
            return choice.message.content  # type: ignore[return-value]
        return getattr(choice, "text", "")


translator = SimpleLLMClient()
translated_field_count = 0
translated_item_indexes: set[int] = set()

print(f"model: {DEFAULT_MODEL}, length: {len(english_candidates)}")
counter = 0

for row in tqdm(english_candidates):
    item_index = int(row["item_index"])  # type:ignore
    source_text = str(row["agent_response"])

    successed = False
    retry = 0
    translated_text = ""
    while not successed and retry < 3:
        try:
            translated_text = translator.ask(source_text, history=HISTORY_SHOT).strip()
            successed = True
            retry = 0
        except Exception as e:
            print(f"Warning: Encounter Error: {e} retry: {retry}")
            retry += 1

    if retry == 3 or not translated_text:
        print(f"Error: Skip {source_text}")
        translated_text = "<error>"

    if translated_text:
        anime_datasets[item_index]["agent_response"] = translated_text
        translated_field_count += 1
        translated_item_indexes.add(item_index)

    # debug model output
    if counter < 3:
        print(f"output #{counter}: {source_text} -> {translated_text}")
        counter += 1

model: qwen3.5-27b, length: 1504


  0%|          | 1/1504 [00:02<1:13:12,  2.92s/it]

output #0: Ah, Raj, your attempt at humor is as misguided as your assumption that bagels are exclusive to Mumbai. Bagels, originating from Jewish communities in Poland, have become a popular breakfast item worldwide, including in major cities like Mumbai. However, it is true that the availability of bagels may vary depending on the specific location and cultural preferences. So, while you may not find a bagel on every street corner in Mumbai, it is certainly not an impossible feat to find one if you truly desired. Now, if you'll excuse me, I need to recalibrate my sarcasm detector after that failed attempt at wit. -> 啊，拉吉，你的幽默尝试与其说是对孟买贝果独占性的误解一样荒谬。贝果起源于波兰的犹太社区，如今已成为全球流行的早餐食品，包括孟买这样的大都市也不例外。当然，贝果的可获得性确实会因具体地点和文化偏好而异。因此，虽然你未必能在孟买的每个街角都找到贝果，但只要你真心想找，也并非不可能。现在，如果各位不介意，我需要重新校准我的讽刺探测器，毕竟刚才那番拙劣的机智表演实在令人难以忍受。


  0%|          | 2/1504 [00:03<40:11,  1.61s/it]  

output #1: A more plausible explanation is that his work in robotics has made an amazing leap forward. -> 更合乎逻辑的解释是，他在机器人领域的研究取得了惊人的飞跃。


...

output #2: Inefficiency. -> 效率低下。


  4%|▍         | 60/1504 [01:18<23:45,  1.01it/s]

  4%|▍         | 61/1504 [01:20<36:10,  1.50s/it]

Error: Skip Well, today we tried masturbating for money.


100%|██████████| 1504/1504 [42:40<00:00,  1.70s/it]  


### Save to local file

In [ ]:
OUTPUT_DATASET_PATH.write_text(
    "\n".join([json.dumps(sample, ensure_ascii=False) for sample in anime_datasets]),
    encoding="utf-8",
)

print(f"English fields detected: {len(english_candidates)}")
print(f"Translated fields: {translated_field_count}")
print(f"Translated items: {len(translated_item_indexes)}")
print(f"Saved to: {OUTPUT_DATASET_PATH}")

English fields detected: 1504
Translated fields: 1504
Translated items: 1504
Saved to: data\raw\Haruhi\Haruhi_clean.jsonl


: 

## Wukong Cleaning Workflow

### Extract corpus content from the file

In [10]:
import json
import os
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple
from dotenv import load_dotenv

from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam as ChatMsgParam

ORIGIN_WUKONG_DATA_PATH = Path(r"..\Dataset\wukong\title_text_embed.jsonl")
OUTPUT_WUKONG_PRE_CLEAN_PATH = Path(r".\data\raw\wukong\wukong_dialog_qa_preclean.jsonl")
OUTPUT_WUKONG_DIALOG_PATH = Path(r".\data\raw\wukong\wukong_dialog_qa.jsonl")
OUTPUT_WUKONG_AUDIT_PATH = Path(r".\data\raw\wukong\wukong_dialog_qa_audit.jsonl")

MIN_TEXT_LEN = 10
MAX_TEXT_LEN = 120

SCENE_PREFIXES = ("scene:", "scene：")
WUKONG_KEYWORDS = (
    "悟空",
    "孙悟空",
    "行者",
    "大圣",
    "齐天大圣",
    "美猴王",
    "老孙",
    "鎮熺┖",
    "瀛欐偀绌",
)
WUKONG_AGENT_ALIASES = {
    "悟空",
    "孙悟空",
    "行者",
    "大圣",
    "齐天大圣",
    "美猴王",
    "老孙",
    "猴哥",
    "孙行者",
    "鎮熺┖",
    "瀛欐偀绌",
}

FORBIDDEN_RESPONSE_TERMS = (
    "泼猴",
    "大圣",
    "悟空",
    "老娘",
    "你这猴",
    "猴头",
)

NARRATION_MARKERS = (
    "长老",
    "八戒",
    "沙僧",
    "唐僧",
    "如来",
    "妖怪",
    "行者道",
    "众人",
    "旁白",
)

DIALOGUE_PATTERN = re.compile(r"^\s*([^:：]{1,24})\s*[:：]\s*(.+?)\s*$")
SPEAKER_TAG_IN_TEXT = re.compile(r"[^\s:：]{1,12}[：:]")
QUOTE_CHARS = "\"'“”‘’「」『』《》〈〉【】（）()"

load_dotenv()

ENABLE_LLM_DYNAMIC_FILTER = os.getenv("WUKONG_ENABLE_LLM_FILTER", "1") == "1"
DYNAMIC_SAMPLE_LIMIT_RAW = os.getenv("WUKONG_DYNAMIC_LIMIT", "0")
DYNAMIC_SAMPLE_LIMIT = int(DYNAMIC_SAMPLE_LIMIT_RAW) if DYNAMIC_SAMPLE_LIMIT_RAW.isdigit() else 0

DEFAULT_API_KEY = os.getenv("OPENAI_API_KEY", "sk-PLACEHOLDER")
DEFAULT_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1")
DEFAULT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "qwen3-max-2025-09-23")

JUDGE_SYSTEM_PROMPT = (
    "你是《西游记》台词数据清洗专家。给定一组对话中的 user_question 与 agent_response，判断 agent_response 是否应被视为孙悟空本人说的话。"
    "若回复更像旁白、多人混合、他人称呼悟空（如泼猴/大圣/悟空）、或明显非悟空第一人称表达，则判定 false。"
    "只输出 JSON：{\"is_wukong\": true/false, \"reason\": \"...\"}，原因不多于 100 字，不要输出其他内容。"
)

ConversationTurn = Tuple[str, Optional[str]]


def _build_messages(
    prompt: str,
    history: Optional[Sequence[ConversationTurn]] = None,
    system_prompt: Optional[str] = None,
) -> list[ChatMsgParam]:
    messages: list[ChatMsgParam] = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    if history:
        for user_msg, assistant_msg in history:
            messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": prompt})
    return messages


@dataclass(slots=True)
class SimpleLLMClient:
    model: str = DEFAULT_MODEL
    api_key: str = DEFAULT_API_KEY
    base_url: Optional[str] = DEFAULT_BASE_URL
    system_prompt: Optional[str] = JUDGE_SYSTEM_PROMPT
    extra_headers: Optional[Dict[str, str]] = None
    _client: OpenAI = field(init=False, repr=False)

    def __post_init__(self) -> None:
        self._client = OpenAI(
            api_key=self.api_key,
            base_url=self.base_url,
            default_headers=self.extra_headers if self.extra_headers else None,
        )

    def ask(
        self,
        prompt: str,
        history: Optional[Sequence[ConversationTurn]] = None,
        *,
        temperature: float = 0.0,
        top_p: float = 0.9,
        max_tokens: Optional[int] = 256,
        **kwargs: Any,
    ) -> str:
        messages = _build_messages(prompt=prompt, history=history, system_prompt=self.system_prompt)
        response = self._client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
            stream=False,
            extra_body={"enable_thinking": False},
            **kwargs,
        )
        choice = response.choices[0]
        if hasattr(choice, "message") and getattr(choice.message, "content", None):
            return choice.message.content  # type: ignore[return-value]
        return getattr(choice, "text", "")


def recover_text(text: str) -> str:
    for enc in ("gbk", "gb18030"):
        try:
            recovered = text.encode(enc).decode("utf-8")
            recovered_cjk = sum("\u4e00" <= ch <= "\u9fff" for ch in recovered)
            original_cjk = sum("\u4e00" <= ch <= "\u9fff" for ch in text)
            if recovered_cjk >= original_cjk:
                return recovered
        except Exception:
            continue
    return text


def strip_outer_quotes(text: str) -> str:
    text = text.strip()
    while text and text[0] in QUOTE_CHARS:
        text = text[1:].strip()
    while text and text[-1] in QUOTE_CHARS:
        text = text[:-1].strip()
    return text


def normalize_speaker(speaker: str) -> str:
    sp = strip_outer_quotes(speaker)
    if sp in WUKONG_AGENT_ALIASES:
        return "孙悟空"
    if any(alias in sp for alias in WUKONG_AGENT_ALIASES):
        return "孙悟空"
    return sp


def extract_turns(text: str) -> list[tuple[str, str]]:
    turns: list[tuple[str, str]] = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.lower().startswith(SCENE_PREFIXES):
            continue
        m = DIALOGUE_PATTERN.match(line)
        if not m:
            continue
        speaker = normalize_speaker(m.group(1).strip())
        utterance = strip_outer_quotes(m.group(2).strip())
        if len(utterance) < MIN_TEXT_LEN:
            continue
        turns.append((speaker, utterance))
    return turns


def is_wukong_related(*parts: str) -> bool:
    blob = "".join(parts)
    return any(keyword in blob for keyword in WUKONG_KEYWORDS)


def static_validate_sample(user_question: str, agent_response: str) -> tuple[bool, str]:
    if len(user_question) < MIN_TEXT_LEN or len(agent_response) < MIN_TEXT_LEN:
        return False, "too_short"
    if len(agent_response) > MAX_TEXT_LEN:
        return False, "too_long"
    if SPEAKER_TAG_IN_TEXT.search(agent_response):
        return False, "contains_speaker_tag"
    if any(term in agent_response for term in FORBIDDEN_RESPONSE_TERMS):
        return False, "forbidden_addressing_term"
    if any(marker in agent_response for marker in NARRATION_MARKERS):
        return False, "contains_narration_marker"
    if user_question == agent_response:
        return False, "response_same_as_question"
    return True, "ok"


def build_judge_prompt(user_question: str, agent_response: str) -> str:
    return (
        "请判断下列 agent_response 是否是孙悟空本人说的话。\n"
        f"user_question: {user_question}\n"
        f"agent_response: {agent_response}\n"
        "只输出 JSON。"
    )


def parse_judge_output(raw_text: str) -> tuple[bool, str]:
    text = raw_text.strip()
    try:
        obj = json.loads(text)
        return bool(obj.get("is_wukong", False)), str(obj.get("reason", ""))
    except Exception:
        pass
    left = text.find("{")
    right = text.rfind("}")
    if 0 <= left < right:
        try:
            obj = json.loads(text[left:right + 1])
            return bool(obj.get("is_wukong", False)), str(obj.get("reason", ""))
        except Exception:
            pass
    print(f"Warning: invalid judge output: {raw_text}")
    return False, "invalid_judge_json"


def llm_dynamic_judge(
    judge_client: SimpleLLMClient,
    user_question: str,
    agent_response: str,
    retries: int = 2,
) -> tuple[bool, str, str]:
    last_raw = ""
    for _ in range(retries + 1):
        try:
            raw = judge_client.ask(
                prompt=build_judge_prompt(user_question, agent_response),
                history=None,
                temperature=0.0,
                max_tokens=256,
            ).strip()
            last_raw = raw
            ok, reason = parse_judge_output(raw)
            return ok, reason, raw
        except Exception as e:
            last_raw = f"judge_error: {e}"
            print(f"Warning: judge call error: {e}, retries left: {retries}")
    print(f"Error: judge call failed after retries, last response: {last_raw}")
    return False, "judge_call_failed", last_raw

### Static Pre-Cleaning (Rule-Based)

In [11]:
if not ORIGIN_WUKONG_DATA_PATH.exists():
    raise FileNotFoundError(f"[Wukong-QA] File not found: {ORIGIN_WUKONG_DATA_PATH}")

base_candidates: list[dict[str, str]] = []
preclean_candidates: list[dict[str, str]] = []
seen_pairs: set[tuple[str, str]] = set()

raw_line_count = 0
parsed_turn_count = 0
wukong_agent_turn_count = 0
static_reject_count = 0

with ORIGIN_WUKONG_DATA_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        raw_line_count += 1
        sample = json.loads(line)

        if sample.get("luotuo_openai") in {"system_prompt", "config"}:
            continue

        text = sample.get("text", "")
        if not isinstance(text, str):
            continue

        text = recover_text(text)
        turns = extract_turns(text)
        parsed_turn_count += len(turns)

        for i in range(1, len(turns)):
            user_speaker, user_question = turns[i - 1]
            agent_speaker, agent_response = turns[i]

            if agent_speaker != "孙悟空":
                continue
            if not is_wukong_related(user_speaker, user_question, agent_speaker, agent_response):
                continue

            user_question = strip_outer_quotes(user_question)
            agent_response = strip_outer_quotes(agent_response)
            pair_key = (user_question, agent_response)
            if pair_key in seen_pairs:
                continue
            seen_pairs.add(pair_key)
            wukong_agent_turn_count += 1

            item = {
                "user_question": user_question,
                "agent_response": agent_response,
                "user_speaker": user_speaker,
                "agent_speaker": agent_speaker,
            }
            base_candidates.append(item)

for item in base_candidates:
    ok, reason = static_validate_sample(item["user_question"], item["agent_response"])
    if not ok:
        static_reject_count += 1
        continue
    preclean_candidates.append(item)

OUTPUT_WUKONG_PRE_CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_WUKONG_PRE_CLEAN_PATH.write_text(
    "\n".join(
        json.dumps(
            {"user_question": i["user_question"], "agent_response": i["agent_response"]},
            ensure_ascii=False,
        )
        for i in preclean_candidates
    ),
    encoding="utf-8",
)

print(f"[Stage-2] Raw lines: {raw_line_count}")
print(f"[Stage-2] Parsed dialogue turns: {parsed_turn_count}")
print(f"[Stage-2] Wukong agent turns (base): {wukong_agent_turn_count}")
print(f"[Stage-2] Base candidates: {len(base_candidates)}")
print(f"[Stage-2] Static kept: {len(preclean_candidates)} | static rejected: {static_reject_count}")
print(f"[Stage-2] Preclean save to: {OUTPUT_WUKONG_PRE_CLEAN_PATH}")

[Stage-2] Raw lines: 947
[Stage-2] Parsed dialogue turns: 5112
[Stage-2] Wukong agent turns (base): 1856
[Stage-2] Base candidates: 1856
[Stage-2] Static kept: 1507 | static rejected: 349
[Stage-2] Preclean save to: data\raw\wukong\wukong_dialog_qa_preclean.jsonl


### Stage 3: Dynamic Tone Filtering (LLM-Based) + Save Outputs

In [9]:
from concurrent.futures import ThreadPoolExecutor, as_completed

from tqdm import tqdm

DYNAMIC_MAX_WORKERS_RAW = os.getenv("WUKONG_DYNAMIC_MAX_WORKERS", "6")
DYNAMIC_MAX_WORKERS = int(DYNAMIC_MAX_WORKERS_RAW) if DYNAMIC_MAX_WORKERS_RAW.isdigit() else 6

final_candidates: list[dict[str, str]] = []
audit_records: list[dict[str, str]] = []
dynamic_reject_count = 0

if ENABLE_LLM_DYNAMIC_FILTER:
    dynamic_inputs = preclean_candidates
    if DYNAMIC_SAMPLE_LIMIT > 0:
        dynamic_inputs = preclean_candidates[:DYNAMIC_SAMPLE_LIMIT]

    judge = SimpleLLMClient()
    ordered_results: list[dict[str, Any]] = [None] * len(dynamic_inputs)  # type: ignore[list-item]

    def _worker(idx: int, item: dict[str, str]) -> tuple[int, bool, str, str, dict[str, str]]:
        keep, reason, judge_raw = llm_dynamic_judge(
            judge_client=judge,
            user_question=item["user_question"],
            agent_response=item["agent_response"],
        )
        return idx, keep, reason, judge_raw, item

    with ThreadPoolExecutor(max_workers=max(1, DYNAMIC_MAX_WORKERS)) as executor:
        futures = [
            executor.submit(_worker, idx, item)
            for idx, item in enumerate(dynamic_inputs)
        ]

        for future in tqdm(as_completed(futures), total=len(futures), desc="Wukong dynamic tone check"):
            idx, keep, reason, judge_raw, item = future.result()
            ordered_results[idx] = {
                "keep": keep,
                "reason": reason,
                "judge_raw": judge_raw,
                "item": item,
            }

    for result in ordered_results:
        if result is None:
            continue
        item = result["item"]
        if result["keep"]:
            final_candidates.append({
                "user_question": item["user_question"],
                "agent_response": item["agent_response"],
            })
            audit_records.append({
                **item,
                "stage": "dynamic",
                "kept": "true",
                "reason": result["reason"],
                "judge_raw": result["judge_raw"],
            })
        else:
            dynamic_reject_count += 1
            audit_records.append({
                **item,
                "stage": "dynamic",
                "kept": "false",
                "reason": result["reason"],
                "judge_raw": result["judge_raw"],
            })
else:
    final_candidates = [
        {"user_question": item["user_question"], "agent_response": item["agent_response"]}
        for item in preclean_candidates
    ]

OUTPUT_WUKONG_DIALOG_PATH.write_text(
    "\n".join(json.dumps(i, ensure_ascii=False) for i in final_candidates),
    encoding="utf-8",
)
OUTPUT_WUKONG_AUDIT_PATH.write_text(
    "\n".join(json.dumps(i, ensure_ascii=False) for i in audit_records),
    encoding="utf-8",
)

if ENABLE_LLM_DYNAMIC_FILTER:
    print(
        f"[Stage-3] Dynamic kept: {len(final_candidates)} | "
        f"dynamic rejected: {dynamic_reject_count} | workers: {DYNAMIC_MAX_WORKERS}"
    )
    if DYNAMIC_SAMPLE_LIMIT > 0:
        print(f"[Stage-3] Dynamic sample limit: {DYNAMIC_SAMPLE_LIMIT}")
else:
    print("[Stage-3] Dynamic filter disabled, final == preclean")

print(f"[Stage-3] Final save to: {OUTPUT_WUKONG_DIALOG_PATH}")
print(f"[Stage-3] Audit save to: {OUTPUT_WUKONG_AUDIT_PATH}")

Wukong dynamic tone check:   1%|          | 17/1507 [00:03<03:00,  8.25it/s]

Wukong dynamic tone check:  13%|█▎        | 197/1507 [00:29<03:34,  6.10it/s]

Wukong dynamic tone check:  20%|█▉        | 299/1507 [00:43<02:42,  7.42it/s]

Wukong dynamic tone check:  45%|████▌     | 685/1507 [01:37<01:42,  8.02it/s]

Wukong dynamic tone check:  53%|█████▎    | 804/1507 [01:55<02:24,  4.88it/s]

Wukong dynamic tone check:  58%|█████▊    | 881/1507 [02:07<01:59,  5.24it/s]

Wukong dynamic tone check:  59%|█████▊    | 884/1507 [02:07<02:01,  5.13it/s]

Wukong dynamic tone check: 100%|██████████| 1507/1507 [03:37<00:00,  6.93it/s]

[Stage-3] Dynamic kept: 700 | dynamic rejected: 807 | workers: 6
[Stage-3] Final save to: data\raw\wukong\wukong_dialog_qa.jsonl
[Stage-3] Audit save to: data\raw\wukong\wukong_dialog_qa_audit.jsonl
